In [ ]:
!pip install kaggle

In [ ]:
#importing dependenties
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Embedding , LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
#data collection
kaggle_dictionary = json.load(open('/content/kaggle.json'))


In [ ]:
#setup kaggle credentials as environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = kaggle_dictionary['key']

In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
  0% 0.00/25.7M [00:00<?, ?B/s]
100% 25.7M/25.7M [00:00<00:00, 489MB/s]


In [ ]:
!ls


imdb-dataset-of-50k-movie-reviews.zip  kaggle.json  sample_data


In [ ]:
#unzip the data set file
with ZipFile("/content/imdb-dataset-of-50k-movie-reviews.zip","r") as zip:
  zip.extractall()

In [ ]:
!ls


'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


In [ ]:
#loading the dataset
Data = pd.read_csv("/content/IMDB Dataset.csv")
Data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
Data.shape

(50000, 2)

In [ ]:
Data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
#replacing +ive to 1 ,-ve to 0
Data.replace({"positive":1,"negative":0},inplace=True)

/tmp/ipython-input-831953623.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Data.replace({"positive":1,"negative":0},inplace=True)


In [ ]:
Data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [ ]:
#splaiiting data into traning data and test data
train_data,test_data = train_test_split(Data,test_size=0.2,random_state=42)

In [ ]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


In [ ]:
#data preprocessing
#tokenizer text data
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]),maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]),maxlen=200)

In [ ]:
print(X_train)


[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [ ]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [ ]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [ ]:
print(Y_train)


39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


In [ ]:
print (Y_test)

33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64


In [ ]:
from tensorflow.keras.layers import Dropout

In [ ]:
#LSTM LongShort-TermMemory
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

In [ ]:
#comple the model
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
#traning the model
model.fit(X_train, Y_train, batch_size=64, epochs=10, validation_split=0.2)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 210s 399ms/step - accuracy: 0.7182 - loss: 0.5377 - val_accuracy: 0.8561 - val_loss: 0.3478
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 200s 400ms/step - accuracy: 0.8311 - loss: 0.3930 - val_accuracy: 0.8486 - val_loss: 0.3602
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 201s 402ms/step - accuracy: 0.8783 - loss: 0.2978 - val_accuracy: 0.8706 - val_loss: 0.3128
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 207s 413ms/step - accuracy: 0.8936 - loss: 0.2656 - val_accuracy: 0.8419 - val_loss: 0.3678
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 200s 401ms/step - accuracy: 0.8986 - loss: 0.2538 - val_accuracy: 0.8676 - val_loss: 0.3219
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 209s 414ms/step - accuracy: 0.9210 - loss: 0.2101 - val_accuracy: 0.8709 - val_loss: 0.3573
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 202s 405ms/step - accuracy: 0.9261 - loss: 0.1902 - val_accuracy: 0.8796 - val_loss: 0.3504
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 261s 402ms/step - accuracy: 0.9326 -

In [ ]:
#model Evaluation
loss, accuracy = model.evaluate(X_test, Y_test)
print(f" Test Loss: {loss}")
print(f"Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 39s 123ms/step - accuracy: 0.8795 - loss: 0.3586
 Test Loss: 0.35225847363471985
Accuracy: 0.8833000063896179


In [ ]:
from typing import Sequence
#predicting system
def predict_sentiment(review):
  #tokenize nad pad it :
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence,maxlen=200)

  #predict the sentiment
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [ ]:
#example
new_review = "this movie wwas fantastic , i loved it."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
The sentiment of the review is: positive


In [ ]:
#example
new_review = "this movie wasn't that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
The sentiment of the review is: negative
